# Week 5: Optimization Improvements + PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/05/Week_05_Optimization_Improvements_PyTorch.ipynb)

**Course:** Neural Architectures and Representation Learning (Master level)

## Learning goals

- Connect the manual training loop from Weeks 3-4 to the PyTorch training loop.
- Understand **momentum** as a smoother update rule than plain SGD.
- Use `loss.backward()` and `optimizer.step()` without treating them as magic.
- Compare **SGD**, **SGD with momentum**, **RMSprop**, and **Adam** on the same small problem.
- Read optimizer behavior from **loss curves** instead of only final numbers.
- Add a practical **early stopping** loop using validation loss and patience.

**Course habit:** change one thing -> run -> observe -> explain.

---

## Environment

**Dependencies:** `torch`, `numpy`, `matplotlib`. CPU is enough.

### Local (uv)

From the repo root:

```bash
uv sync
uv run jupyter notebook weeks/05/Week_05_Optimization_Improvements_PyTorch.ipynb
```

### Colab

1. Open the notebook via the badge above.
2. Runtime -> Run all.
3. Colab normally includes PyTorch already. Do not upgrade packages unless needed.

In [ ]:
# Optional update cell
# Run this only if your runtime has an old PyTorch version,
# or if your instructor explicitly asks you to update.
#
# Important:
# 1. Run this before importing torch.
# 2. After installation, restart the runtime/kernel.
# 3. Then run the notebook again from the top.
#
# %pip install --upgrade torch torchvision

In [ ]:
import copy
import math

import numpy as np
import torch
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    plt.rc("axes", grid=True)

%matplotlib inline

print("torch:", torch.__version__)
print("numpy:", np.__version__)

major = int(torch.__version__.split(".")[0])
assert major >= 2, "This notebook expects PyTorch 2.x or newer."

device = torch.device("cpu")
print("device:", device)

---

## 1. Quick recap: the update rule

Last week, the whole training step could be summarized as:

$$w \leftarrow w - \eta \nabla L(w)$$

The learning rate $\eta$ controls step size. Momentum adds a velocity:

$$v \leftarrow \beta v + \nabla L(w), \qquad w \leftarrow w - \eta v$$

Here `beta` decides how much previous direction is remembered.

### Mini task

Change `learning_rate` and `beta` below. What happens when beta is `0.0`, `0.8`, `0.95`?

In [ ]:
def run_quadratic_descent(learning_rate=0.1, beta=0.0, n_steps=50):
    # Minimize L(w) = (w - 3)^2 with optional momentum.
    w = 0.0
    velocity = 0.0
    losses = []
    weights = []

    for _ in range(n_steps):
        loss = (w - 3.0) ** 2
        grad = 2.0 * (w - 3.0)
        velocity = beta * velocity + grad
        w = w - learning_rate * velocity
        losses.append(loss)
        weights.append(w)

    return np.array(losses), np.array(weights)


learning_rate = 0.15  # TODO: try 0.03, 0.15, 0.6
beta = 0.0            # TODO: try 0.0, 0.8, 0.95

losses, weights = run_quadratic_descent(learning_rate=learning_rate, beta=beta)

print(f"final w = {weights[-1]:.4f}, final loss = {losses[-1]:.6f}")

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.plot(np.maximum(losses, 1e-20), "o-", markersize=3)
ax.set_yscale("log")
ax.set_xlabel("Step")
ax.set_ylabel("Loss")
ax.set_title(f"Quadratic loss: lr={learning_rate}, beta={beta}")
ax.grid(True, which="both", alpha=0.35)
plt.tight_layout()
plt.show()

---

## 2. Momentum, refined

Momentum is not a different loss function. It changes the **update direction** by mixing the current gradient with a running direction from previous steps.

**Common confusion**

- Gradient: direction from the current step.
- Velocity: running blend of current and previous gradients.
- Loss: the score we are trying to minimize. We do not multiply the loss by beta.

### Demo: same quadratic, different beta values

In [ ]:
betas = [0.0, 0.8, 0.9]
learning_rate = 0.08

plt.figure(figsize=(7, 4))
for beta in betas:
    losses, _ = run_quadratic_descent(learning_rate=learning_rate, beta=beta, n_steps=60)
    label = "plain SGD" if beta == 0.0 else f"momentum beta={beta}"
    plt.plot(np.maximum(losses, 1e-20), label=label)

plt.xlabel("Step")
plt.ylabel("Loss")
plt.yscale("log")
plt.title("Momentum changes the path of the updates")
plt.grid(True, which="both", alpha=0.35)
plt.legend()
plt.tight_layout()
plt.show()

**Pause and reflect**

1. Does higher beta always mean better?
2. What could go wrong if learning rate and beta are both large?
3. Why do we compare curves instead of only final loss?

---

## Shared setup: a tiny regression problem

We now move from one scalar parameter to a small neural network.

The data is synthetic:

$$y = \sin(2x) + 0.3x + \epsilon$$

This is deliberately small, visual, and CPU-friendly. The goal is not a benchmark; the goal is to compare optimizer behavior.

In [ ]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_regression_data(n_samples=160, noise=0.08, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-2.5, 2.5, size=(n_samples, 1)).astype(np.float32)
    y = np.sin(2.0 * x) + 0.3 * x + rng.normal(0.0, noise, size=(n_samples, 1)).astype(np.float32)

    order = rng.permutation(n_samples)
    x = x[order]
    y = y[order]

    split = int(0.75 * n_samples)
    X_train = torch.tensor(x[:split], device=device)
    y_train = torch.tensor(y[:split], device=device)
    X_val = torch.tensor(x[split:], device=device)
    y_val = torch.tensor(y[split:], device=device)
    return X_train, y_train, X_val, y_val


def make_model(seed=42, hidden=24):
    set_seed(seed)
    return torch.nn.Sequential(
        torch.nn.Linear(1, hidden),
        torch.nn.Tanh(),
        torch.nn.Linear(hidden, 1),
    ).to(device)


X_train, y_train, X_val, y_val = make_regression_data(seed=4)

plt.figure(figsize=(6, 3))
plt.scatter(X_train.numpy(), y_train.numpy(), s=18, alpha=0.75, label="train")
plt.scatter(X_val.numpy(), y_val.numpy(), s=28, alpha=0.9, marker="x", label="val")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Tiny synthetic regression dataset")
plt.grid(True, alpha=0.35)
plt.legend()
plt.tight_layout()
plt.show()

---

## 3. PyTorch transition: manual loop -> framework loop

The loop is still the same:

1. Forward pass: compute predictions.
2. Loss: compare predictions to targets.
3. Backward pass: compute gradients.
4. Optimizer step: update parameters.
5. Zero gradients: clear old gradients before the next step.

The new part is that PyTorch computes gradients automatically.

In [ ]:
model = make_model(seed=1)
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# One training step, written slowly.
y_pred = model(X_train)
loss = loss_fn(y_pred, y_train)

print("Loss before backward:", float(loss))
print("Gradient before backward:", model[0].weight.grad)

loss.backward()
print("Gradient norm after backward:", float(model[0].weight.grad.norm()))

optimizer.step()
optimizer.zero_grad()

print("Gradient after zero_grad:", model[0].weight.grad)

**Mapping to our manual code**

- `model(X_train)` replaces our hand-written `forward`.
- `loss.backward()` replaces our hand-written `backward`.
- `optimizer.step()` replaces `W -= lr * dW`.
- `optimizer.zero_grad()` prevents gradients from accumulating across steps.

In [ ]:
def build_optimizer(name, parameters, lr=0.01, momentum=0.0):
    name = name.lower()
    if name == "sgd":
        return torch.optim.SGD(parameters, lr=lr)
    if name == "momentum":
        return torch.optim.SGD(parameters, lr=lr, momentum=momentum)
    if name == "rmsprop":
        return torch.optim.RMSprop(parameters, lr=lr)
    if name == "adam":
        return torch.optim.Adam(parameters, lr=lr)
    raise ValueError(f"Unknown optimizer: {name}")


def train_torch_optimizer(
    optimizer_name="sgd",
    lr=0.01,
    momentum=0.0,
    n_epochs=250,
    seed=42,
    hidden=24,
):
    X_train, y_train, X_val, y_val = make_regression_data(seed=4)
    model = make_model(seed=seed, hidden=hidden)
    loss_fn = torch.nn.MSELoss()
    optimizer = build_optimizer(optimizer_name, model.parameters(), lr=lr, momentum=momentum)

    train_losses = []
    val_losses = []

    for _ in range(n_epochs):
        model.train()
        y_pred = model(X_train)
        train_loss = loss_fn(y_pred, y_train)

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = loss_fn(val_pred, y_val)

        train_losses.append(float(train_loss.detach()))
        val_losses.append(float(val_loss.detach()))

    return {
        "model": model,
        "train_losses": np.array(train_losses),
        "val_losses": np.array(val_losses),
    }


def plot_curves(curves, title, ylabel="Train MSE"):
    plt.figure(figsize=(8, 4))
    linestyles = ["-", "--", "-.", ":"]
    for i, (label, values) in enumerate(curves):
        plt.plot(np.maximum(values, 1e-20), linestyle=linestyles[i % len(linestyles)], label=label)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.yscale("log")
    plt.title(title)
    plt.grid(True, which="both", alpha=0.35)
    plt.legend()
    plt.tight_layout()
    plt.show()

---

## 4. Coding block 1: SGD vs momentum (about 40 min)

**Goal:** compare plain SGD and momentum on the same model and data.

**Core path**

1. Edit `lr_sgd`, `lr_momentum`, and `beta_momentum`.
2. Keep the same `seed` for a fair comparison.
3. Plot both curves.
4. Write down which curve is faster, smoother, and more stable.

**Important:** same optimizer with a bad learning rate can still fail. Momentum is not magic.

In [ ]:
# TODO: edit these values and re-run.
lr_sgd = 0.03
lr_momentum = 0.03
beta_momentum = 0.9

n_epochs = 250
seed = 7

sgd_run = train_torch_optimizer("sgd", lr=lr_sgd, n_epochs=n_epochs, seed=seed)
mom_run = train_torch_optimizer("momentum", lr=lr_momentum, momentum=beta_momentum, n_epochs=n_epochs, seed=seed)

plot_curves(
    [
        (f"SGD lr={lr_sgd}", sgd_run["train_losses"]),
        (f"Momentum lr={lr_momentum}, beta={beta_momentum}", mom_run["train_losses"]),
    ],
    "Coding block 1: SGD vs momentum",
)

print("Final SGD train loss:", sgd_run["train_losses"][-1])
print("Final momentum train loss:", mom_run["train_losses"][-1])

### Try a beta sweep

Change `beta_values` and re-run. Typical momentum values are around `0.8` to `0.95`.

In [ ]:
# TODO: edit this list.
beta_values = [0.0, 0.5, 0.8, 0.9]
learning_rate = 0.03

curves = []
for beta in beta_values:
    if beta == 0.0:
        run = train_torch_optimizer("sgd", lr=learning_rate, n_epochs=250, seed=7)
        label = f"SGD beta={beta}"
    else:
        run = train_torch_optimizer("momentum", lr=learning_rate, momentum=beta, n_epochs=250, seed=7)
        label = f"Momentum beta={beta}"
    curves.append((label, run["train_losses"]))

plot_curves(curves, "Momentum beta sweep")

<details>
<summary>One reasonable starting point</summary>

```python
lr_sgd = 0.03
lr_momentum = 0.03
beta_momentum = 0.9
beta_values = [0.0, 0.5, 0.8, 0.9]
```

Then try a larger learning rate and watch whether momentum helps or hurts.

</details>

---

## 5. Adaptive optimizers

Momentum smooths the **direction** of updates. Adaptive optimizers also change the **step size per parameter**.

### RMSprop

RMSprop tracks a running average of squared gradients. Parameters with consistently large gradients get smaller effective steps.

### Adam

Adam combines two ideas:

- momentum-like smoothing of gradients
- adaptive scaling based on squared gradients

**Intuition:** Adam often works well with less tuning, but it is still not a substitute for understanding learning rate, validation curves, and overfitting.

### Optimizer Overview

| Optimizer | Uses gradients | Smooths direction | Adapts step size | Typical behavior |
|-----------|----------------|-------------------|------------------|------------------|
| SGD | ✓ | ✗ | ✗ | Simple baseline; sensitive to learning rate |
| Momentum | ✓ | ✓ | ✗ | Often smoother than SGD; can overshoot if tuned badly |
| RMSprop | ✓ | ✗ | ✓ | Adjusts per-parameter steps; useful when gradients have uneven scale |
| Adam | ✓ | ✓ | ✓ | Fast default choice; still needs validation and learning-rate awareness |

- **SGD:** same step rule everywhere.
- **Momentum:** smooths the update direction.
- **RMSprop:** adapts step size per parameter.
- **Adam:** combines momentum and adaptive step sizes.


---

## 6. Optimizer comparison

Now we compare four optimizers on the same data and model initialization:

- SGD
- SGD with momentum
- RMSprop
- Adam

The learning rates below are reasonable starting points, not universal truths.

In [ ]:
comparison_specs = [
    ("SGD lr=0.03", "sgd", 0.03, 0.0),
    ("Momentum lr=0.03", "momentum", 0.03, 0.9),
    ("RMSprop lr=0.01", "rmsprop", 0.01, 0.0),
    ("Adam lr=0.01", "adam", 0.01, 0.0),
]

curves = []
for label, opt_name, lr, momentum in comparison_specs:
    run = train_torch_optimizer(opt_name, lr=lr, momentum=momentum, n_epochs=250, seed=7)
    curves.append((label, run["train_losses"]))

plot_curves(curves, "Optimizer comparison on the same toy task")

**How to read this plot**

- Fast early drop can be useful, but final validation behavior still matters.
- Optimizer comparisons are only fair when data, model, initialization, and epoch budget are controlled.
- The "best" optimizer can change when the learning rate changes.

---

## 7. Coding block 2: optimizer exploration (about 40-45 min)

**Goal:** compare optimizers under both shared and tuned learning rates.

**Core path**

1. Edit `experiments`.
2. First try the same learning rate for all optimizers.
3. Then tune each optimizer separately.
4. Compare speed, stability, and final loss.

**Question:** Which optimizer is most sensitive to learning rate?

In [ ]:
# TODO: edit this list.
# Format: (label, optimizer_name, learning_rate, momentum)
experiments = [
    ("SGD lr=0.01", "sgd", 0.01, 0.0),
    ("Momentum lr=0.01", "momentum", 0.01, 0.9),
    ("RMSprop lr=0.01", "rmsprop", 0.01, 0.0),
    ("Adam lr=0.01", "adam", 0.01, 0.0),
]

curves = []
summary = []
for label, opt_name, lr, momentum in experiments:
    run = train_torch_optimizer(opt_name, lr=lr, momentum=momentum, n_epochs=300, seed=12)
    curves.append((label, run["train_losses"]))
    summary.append((label, run["train_losses"][-1], run["val_losses"][-1]))

plot_curves(curves, "Coding block 2: optimizer exploration")

for label, train_final, val_final in summary:
    print(f"{label:18s} train={train_final:.5f}  val={val_final:.5f}")

### Optional: visualize predictions

Pick one optimizer from your experiment list and inspect the fitted curve.

In [ ]:
chosen_optimizer = "adam"  # TODO: try "sgd", "momentum", "rmsprop", "adam"
chosen_lr = 0.01
chosen_momentum = 0.9

run = train_torch_optimizer(chosen_optimizer, lr=chosen_lr, momentum=chosen_momentum, n_epochs=300, seed=12)
model = run["model"]

x_grid = torch.linspace(-2.5, 2.5, 200).reshape(-1, 1)
with torch.no_grad():
    y_grid = model(x_grid).numpy()

plt.figure(figsize=(7, 4))
plt.scatter(X_train.numpy(), y_train.numpy(), s=18, alpha=0.6, label="train")
plt.scatter(X_val.numpy(), y_val.numpy(), s=28, marker="x", alpha=0.8, label="val")
plt.plot(x_grid.numpy(), y_grid, color="black", linewidth=2, label=f"{chosen_optimizer} prediction")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Model prediction after training")
plt.grid(True, alpha=0.35)
plt.legend()
plt.tight_layout()
plt.show()

---

## 8. Early stopping

Validation loss is a monitoring signal. Early stopping means:

1. Train one epoch.
2. Check validation loss.
3. If validation loss improves, remember the model.
4. If validation loss does not improve for `patience_epochs`, stop.

This prevents unnecessary training and can reduce overfitting.

In [ ]:
def train_with_early_stopping(
    optimizer_name="adam",
    lr=0.01,
    momentum=0.0,
    max_epochs=600,
    patience_epochs=30,
    seed=21,
):
    X_train, y_train, X_val, y_val = make_regression_data(seed=9, noise=0.12)
    model = make_model(seed=seed, hidden=48)
    loss_fn = torch.nn.MSELoss()
    optimizer = build_optimizer(optimizer_name, model.parameters(), lr=lr, momentum=momentum)

    best_val = float("inf")
    best_epoch = 0
    best_state = copy.deepcopy(model.state_dict())
    wait = 0
    train_losses = []
    val_losses = []

    for epoch in range(max_epochs):
        model.train()
        pred = model(X_train)
        train_loss = loss_fn(pred, y_train)

        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val), y_val)

        train_losses.append(float(train_loss.detach()))
        val_losses.append(float(val_loss.detach()))

        if val_losses[-1] < best_val:
            best_val = val_losses[-1]
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if wait >= patience_epochs:
            break

    model.load_state_dict(best_state)
    return model, np.array(train_losses), np.array(val_losses), best_epoch


model_es, train_es, val_es, best_epoch = train_with_early_stopping(
    optimizer_name="adam",
    lr=0.01,
    max_epochs=600,
    patience_epochs=30,
)

plt.figure(figsize=(8, 4))
plt.plot(np.maximum(train_es, 1e-20), label="train")
plt.plot(np.maximum(val_es, 1e-20), label="validation")
plt.axvline(best_epoch, color="black", linestyle="--", linewidth=1.5, label=f"best epoch={best_epoch}")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.yscale("log")
plt.title("Early stopping watches validation loss")
plt.grid(True, which="both", alpha=0.35)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Stopped after {len(train_es)} epochs; best validation epoch was {best_epoch}.")

---

## Wrap-up: takeaways

1. PyTorch automates gradients, but the training loop is still recognizable.
2. Momentum smooths updates by remembering previous gradient directions.
3. RMSprop and Adam adapt step sizes per parameter.
4. Optimizer choice and learning rate must be considered together.
5. Validation curves matter; early stopping turns them into an action.

**Habit:** change one thing -> run -> observe -> explain.

---

## Homework / Post-class Extensions

Optional unless assigned. These are scaffolds for students who want more practice.

| Idea | What to try |
|------|-------------|
| Learning-rate schedule | Reduce `lr` over time |
| Momentum beta search | Sweep beta from `0.0` to `0.99` |
| Optimizer + schedule | Combine Adam or SGD with an LR schedule |
| Early stopping variants | Change patience and minimum improvement |
| Hyperparameter search | Try a small grid over optimizer and learning rate |
| Optional reading | Adam paper: Kingma and Ba (2014) |

### Extension A: Step learning-rate schedule

In [ ]:
def step_lr(epoch, initial_lr=0.05, drop_every=100, gamma=0.5):
    # TODO: return a learning rate that drops by gamma every drop_every epochs.
    # Hint: initial_lr * (gamma ** (epoch // drop_every))
    raise NotImplementedError


# Example after implementation:
# for group in optimizer.param_groups:
#     group["lr"] = step_lr(epoch)

### Extension B: Exponential learning-rate decay

In [ ]:
def exponential_lr(epoch, initial_lr=0.05, gamma=0.99):
    # TODO: return initial_lr * gamma ** epoch
    raise NotImplementedError

### Extension C: Cosine learning-rate schedule

In [ ]:
def cosine_lr(epoch, max_epochs, lr_max=0.05, lr_min=0.001):
    # TODO: implement cosine annealing from lr_max to lr_min.
    # Hint: use math.cos(math.pi * epoch / max_epochs)
    raise NotImplementedError

### Extension D: Small hyperparameter search

In [ ]:
# TODO: loop over optimizers and learning rates.
# Record final validation loss for each combination.
# Print the best combination.

search_space = [
    ("sgd", 0.01, 0.0),
    ("momentum", 0.01, 0.9),
    ("rmsprop", 0.01, 0.0),
    ("adam", 0.01, 0.0),
]

# for optimizer_name, lr, momentum in search_space:
#     ...
pass